In [ ]:
## 1. Make classification data and get it ready
import sklearn
from sklearn.datasets import make_circles
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
n_samples = 1000

X,y = make_circles(n_samples,noise=0.03,random_state=42)



In [ ]:
len(X),len(y)

In [ ]:
print(f"First five samples of X:\n{X[:5]}")
print(f"First five sample of y: {y[:5]}")

In [ ]:
import pandas as pd

circles = pd.DataFrame({"X1": X[:, 0], "X2": X[:, 0],"label":y})
circles.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x = X[:,0],y = X[:,1],c=y,cmap=plt.cm.RdYlBu)

In [ ]:
if(X.dtype == "float64"):
    X = torch.from_numpy(X).type(torch.float32)
    y = torch.from_numpy(y).type(dtype = torch.float32)

X[:5],y[:5]


In [ ]:
# SPlit DATa
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
len(X_train),len(X_test),len(y_train),len(y_test)

In [ ]:
## 1. Building a model
class CircleModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        # 2. Create two nn.Linear
        self.layer_1 = nn.Linear(in_features=2, out_features=5)
        self.layer_2 = nn.Linear(in_features=5, out_features=1)
    def forward(self,X):
        return self.layer_2(self.layer_1(X))
model_0 = CircleModelV0().to(device)

model_0

In [ ]:
model_0 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5), nn.Linear(in_features=5, out_features=1).to(device)
)

In [ ]:
model_0.state_dict()

In [ ]:
with torch.inference_mode():
    untrained_preds = model_0(X_test.to(device))
print(f"Length of prediction: {len(untrained_preds)},shape:{untrained_preds.shape} ")
print(f"Length of test samples:{len(X_test)}, Shape:{X_test.shape}")
print(f"\n First 10 prediction :\n{untrained_preds}")
print(f"\nFirst 10 labels:\n{y_test[:10]}")

In [ ]:
lossfn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

In [ ]:
model_0.state_dict()

In [17]:
# Accuracy
def accuracy_fn(y_true,y_pred):
    correct = torch.eq(y_true,y_pred).sum().item()
    acc = (correct/len(y_pred))*100
    return acc


In [27]:
# 3. Train model
with torch.inference_mode():
    y_logits = model_0(X_test.to(device))[:5]

y_logits

tensor([[ 0.1468],
        [ 0.4438],
        [-0.3537],
        [ 0.2875],
        [ 0.0877]])

In [23]:
y_pred_probs = torch.sigmoid(y_logits)
y_pred_probs

tensor([[0.5366],
        [0.6092],
        [0.4125],
        [0.5714],
        [0.5219]])

In [31]:
# Find the predicted
y_preds = torch.round(y_pred_probs)


y_preds_label = torch.round(torch.sigmoid(model_0(X_test.to(device))[:5]))
print(torch.eq(y_preds.squeeze(),y_preds_label.squeeze()))

tensor([True, True, True, True, True])


8.0